# Jour 2 · De l'erreur de prévision aux anomalies


## Objectifs

- transformer un résidu de prévision en score d'anomalie
- choisir un seuil à partir du comportement d'entraînement
- mesurer faux positifs et anomalies manquées

Si un modèle décrit le comportement habituel, une grande différence entre mesure et prévision peut signaler un événement inhabituel. Ce n'est pas une preuve de panne : c'est un **signal à investiguer**.

![Écarts verticaux entre une courbe observée et une courbe prévue](../assets/jour_02/03_residu_observe_prevu.png)

*Le résidu mesure à chaque instant ce que le modèle n'a pas expliqué : observation moins prévision.*

![Erreur absolue comparée à un seuil avec points d'alerte](../assets/jour_02/03_erreur_seuil_alerte.png)

*Une erreur devient une alerte seulement lorsqu'elle dépasse un seuil appris sur le comportement passé.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import precision_score, recall_score, f1_score


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
df = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_labeled.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
hourly = df.set_index("timestamp").resample("1h").agg({
    "temperature_c": "mean",
    "is_anomaly": "max",
})

cutoff = hourly.index.min() + pd.Timedelta(days=28)
train = hourly.loc[hourly.index < cutoff]
test = hourly.loc[hourly.index >= cutoff]

In [ ]:
prophet_train = train["temperature_c"].rename_axis("ds").rename("y").reset_index()
prophet_train["ds"] = prophet_train["ds"].dt.tz_localize(None)

model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=False,
    uncertainty_samples=0,
).fit(prophet_train)

all_dates = pd.DataFrame({"ds": hourly.index.tz_localize(None)})
all_forecast = model.predict(all_dates)
hourly["forecast"] = all_forecast["yhat"].to_numpy()
hourly["absolute_error"] = (
    hourly["temperature_c"] - hourly["forecast"]
).abs()

In [ ]:
threshold = hourly.loc[hourly.index < cutoff, "absolute_error"].quantile(0.99)
hourly["detected"] = (hourly["absolute_error"] > threshold).astype(int)

evaluation = hourly.loc[hourly.index >= cutoff]
print(f"Seuil appris : {threshold:.2f} °C")
print("Précision :", round(precision_score(evaluation["is_anomaly"], evaluation["detected"], zero_division=0), 3))
print("Rappel    :", round(recall_score(evaluation["is_anomaly"], evaluation["detected"], zero_division=0), 3))
print("F1        :", round(f1_score(evaluation["is_anomaly"], evaluation["detected"], zero_division=0), 3))

In [ ]:
view = hourly.loc[hourly.index >= cutoff]
ax = view["temperature_c"].plot(figsize=(13, 5), label="température", color="black")
view["forecast"].plot(ax=ax, label="prévision", color="tab:blue")
detected = view[view["detected"] == 1]
ax.scatter(detected.index, detected["temperature_c"], color="red", label="alerte", zorder=3)
ax.set_ylabel("°C")
ax.set_title("Une grande erreur de prévision devient une alerte")
ax.legend()
plt.show()

- **Précision** : parmi nos alertes, combien correspondaient à une anomalie ?
- **Rappel** : parmi les anomalies, combien avons-nous détectées ?

Baisser le seuil augmente généralement le rappel, mais aussi les fausses alertes.

![Matrice expliquant vrais positifs, faux positifs, faux négatifs et vrais négatifs](../assets/jour_02/03_precision_et_rappel.png)

*La précision juge la qualité des alertes ; le rappel juge la couverture des anomalies réelles.*

### À vous de jouer — tester plusieurs seuils

Comparez les quantiles 0,98, 0,99 et 0,995. Construisez un tableau avec seuil, nombre d'alertes, précision et rappel sur la période de test.

In [ ]:
# Écrivez votre code ici.
pass

## À retenir

Le seuil est une décision métier autant que statistique. Une alerte manquée et une fausse alerte n'ont pas forcément le même coût.